# LLM setup (gemini)

In [4]:
from google import genai
from IPython.display import Markdown

In [5]:
from dotenv import load_dotenv
load_dotenv()
import os

api_key = os.getenv("GEMINI_API_KEY")

In [7]:
client = genai.Client()
client

In [8]:
def get_response(prompt):

    response = client.models.generate_content(
        model= 'gemini-2.5-flash-lite',
        contents=prompt
    )

    markdown_result = Markdown(response.text)
    # return response.text
    return markdown_result

prompt01 = 'add 33+44'

result = get_response(prompt01)

# Markdown(result)
result

33 + 44 = 77

# Task 1 : Sentiment‑with‑CoT from Customer Reviews

In [ ]:
url = "https://www.amazon.in/Bombay-Shaving-Company-OmniBlade-Detachable/dp/B0F99Q4Q3S/ref=sr_1_2?nsdOptOutParam=true&sr=8-2"

200


Scraped data from amazon using Amazon Review Export Tool

In [5]:
import pandas as pd 

data = pd.read_csv('amazon_reviews_B0F99Q4Q3S.csv')
data.head()

,ID,Username,ProfileURL,Title,Rating,Review Text,Review Content,Review URL,Date,Format,Verified,Helpful,Thumbnails,Images,Video,Scraped At
0,R1GOLXV2NQ3J71,Arun K S,https://www.amazon.in/gp/profile/amzn1.account...,Good product,5,5.0 out of 5 stars,"Nice product, meeting the expectations. Fair p...",https://www.amazon.in/gp/customer-reviews/R1GO...,Reviewed in India on 4 March 2026,NaN,Verified Purchase,NaN,https://m.media-amazon.com/images/I/71VUsP8A42...,https://m.media-amazon.com/images/I/71VUsP8A42...,NaN,2026-05-17T09:04:04.781Z
1,R1XTJRXJMY204V,Tanusree,https://www.amazon.in/gp/profile/amzn1.account...,Worst product,1,1.0 out of 5 stars,After using 3 months it's not a good product w...,https://www.amazon.in/gp/customer-reviews/R1XT...,Reviewed in India on 24 April 2026,NaN,Verified Purchase,One person found this helpful,NaN,NaN,NaN,2026-05-17T09:04:04.781Z
2,R39UY88UZ8W8EY,Iqbal Singh Aulakh,https://www.amazon.in/gp/profile/amzn1.account...,Excellent,5,5.0 out of 5 stars,Excellent,https://www.amazon.in/gp/customer-reviews/R39U...,Reviewed in India on 22 April 2026,NaN,Verified Purchase,NaN,NaN,NaN,NaN,2026-05-17T09:04:04.781Z
3,R1RYJ1I7CUQPY2,Varun K.,https://www.amazon.in/gp/profile/amzn1.account...,Good alternative oh 1blade,4,4.0 out of 5 stars,A great alternative of OneBlade. but Blade rep...,https://www.amazon.in/gp/customer-reviews/R1RY...,Reviewed in India on 17 April 2026,NaN,Verified Purchase,One person found this helpful,NaN,NaN,NaN,2026-05-17T09:04:04.781Z
4,R1CU90LA0XSYF7,Murugesh,https://www.amazon.in/gp/profile/amzn1.account...,Didn’t have long life,3,3.0 out of 5 stars,"Worked well but the tip broke , looks like the...",https://www.amazon.in/gp/customer-reviews/R1CU...,Reviewed in India on 20 February 2026,NaN,Verified Purchase,5 people found this helpful,https://m.media-amazon.com/images/I/61932Mqkkg...,https://m.media-amazon.com/images/I/61932Mqkkg...,NaN,2026-05-17T09:04:04.781Z


In [11]:
df = data['Review Content']
df[4]

'Worked well but the tip broke , looks like the clamp on side broke and was not able to fix it .Used it judiciously, but anyways..'

## Prompt design 

In [ ]:
prompt = """You are a sentiment analysis assistant.

Your task is to classify customer reviews into one of three labels:
- positive
- neutral
- negative

Follow these steps carefully for every review:

1. Identify all positive-sentiment phrases.
2. Identify all negative-sentiment phrases.
3. Check for contradictions or mixed sentiment.
4. Decide the overall sentiment label.
5. Explain clearly why the label was chosen.

--------------------------------------------------
Example 1

Review:
"Excellent product. Battery life is amazing and performance is very smooth. Worth the money."

Step 1 - Positive phrases:
- "Excellent product"
- "Battery life is amazing"
- "performance is very smooth"
- "Worth the money"

Step 2 - Negative phrases:
- None

Step 3 - Mixed sentiment check:
- No contradictions or mixed opinions found.

Step 4 - Final label:
positive

Step 5 - Explanation:
The review contains multiple strong positive opinions about quality, battery life, and value. No negative feedback is mentioned.

--------------------------------------------------
Example 2

Review:
"The product is okay. Build quality is decent but delivery was delayed."

Step 1 - Positive phrases:
- "product is okay"
- "Build quality is decent"

Step 2 - Negative phrases:
- "delivery was delayed"

Step 3 - Mixed sentiment check:
- The review contains both positive and negative points.

Step 4 - Final label:
neutral

Step 5 - Explanation:
The customer mentions both satisfactory and unsatisfactory experiences. Since the sentiment is balanced, the overall label is neutral.

--------------------------------------------------
Example 3

Review:
"Very disappointed. The motor stopped working within a week and customer service was terrible."

Step 1 - Positive phrases:
- None

Step 2 - Negative phrases:
- "Very disappointed"
- "motor stopped working within a week"
- "customer service was terrible"

Step 3 - Mixed sentiment check:
- No positive feedback is present.

Step 4 - Final label:
negative

Step 5 - Explanation:
The review expresses strong dissatisfaction regarding both product quality and customer support.

--------------------------------------------------
Now analyze the following review:

Review:
"{review}"

Provide:
- Positive phrases
- Negative phrases
- Mixed sentiment analysis
- Final label
- Explanation""" 